In [7]:
!export https_proxy=http://10.1.2.1:7890
!export http_proxy=http://10.1.2.1:7890
!pip config set global.index-url https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple

Writing to /usr/pip.conf


In [1]:
import os
import sys
import datasets
sys.path.insert(0, "/gpfs/users/zhangyiqi/srl/verl")
from verl.utils.hdfs_io import copy, makedirs
import argparse

import re

from verl.utils.hdfs_io import copy, makedirs
import argparse

from verl.utils.reward_score.math import remove_boxed, last_boxed_only_string
from verl.workers.reward_manager import DAPORewardManager
from transformers import AutoTokenizer, AutoProcessor
from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn

In [2]:
tokenizer = AutoTokenizer.from_pretrained('/gpfs/models/huggingface.co/meta-llama/Meta-Llama-3___1-8B-Instruct')
processor = AutoProcessor.from_pretrained('/gpfs/models/huggingface.co/meta-llama/Meta-Llama-3___1-8B-Instruct')
dataset = RLHFDataset(
    parquet_files=['/gpfs/users/zhangyiqi/srl/data/dapo-math-17k.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )

dataset len: 1791700


In [5]:
import pyarrow.parquet as pq

data = pq.read_table('/gpfs/users/zhangyiqi/srl/data/dapo-math-17k.parquet')
data['reward_model']

[
  -- is_valid: all not null
  -- child 0 type: string
    [
      "34",
      "113",
      "-3",
      "3",
      "37",
      ...
      "61",
      "750",
      "400",
      "182",
      "480"
    ]
  -- child 1 type: string
    [
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      ...
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2"
    ],
  -- is_valid: all not null
  -- child 0 type: string
    [
      "588",
      "70",
      "112",
      "486",
      "19",
      ...
      "39",
      "8",
      "76",
      "655",
      "41"
    ]
  -- child 1 type: string
    [
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      ...
      "rule-lighteval/MATH_v2",

In [3]:
def extract_solution(solution_str):
    solution = re.search("#### (\\-?[0-9\\.\\,]+)", solution_str)
    assert solution is not None
    final_solution = solution.group(0)
    final_solution = final_solution.split('#### ')[1].replace(',', '')
    return final_solution
    
def prepare_gsm8k():
    data_source = '/gpfs/datasets/openai/gsm8k'

    dataset = datasets.load_dataset(data_source, 'main')

    dataset = dataset['test']

    instruction_following_1 = 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n'
    instruction_following_2 = 'Remember to put your answer on its own line after "Answer:".'
    # add a row to each data item that represents a unique id
    def make_map_fn(split):

        def process_fn(example, idx):
            question_raw = example.pop('question')

            question = instruction_following_1 + '\n' + question_raw + '\n' + instruction_following_2

            answer_raw = example.pop('answer')
            solution = extract_solution(answer_raw)
            data = {
                "data_source": 'math_dapo',
                "prompt": [{
                    "role": "user",
                    "content": question,
                }],
                "ability": "MATH",
                "reward_model": {
                    "style": "rule-lighteval/MATH_v2",
                    "ground_truth": solution
                },
                "extra_info": {
                    'dummy': 'dummy',
                }
            }
            return data

        return process_fn

    dataset = dataset.map(function=make_map_fn('test'), with_indices=True)
    print(f'dataset size: {len(dataset)}')
    return dataset

gsm8k_dataset = prepare_gsm8k()
gsm8k_dataset.to_parquet('/gpfs/users/zhangyiqi/srl/data/gsm8k_eval.parquet')

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

dataset size: 1319


Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

735049

In [23]:
def prepare_math500():
    data_source = '/gpfs/users/zhangyiqi/.cache/huggingface/hub/datasets--HuggingFaceH4--MATH-500'

    dataset = datasets.load_dataset(data_source, 'default')

    dataset = dataset['test']

    instruction_following_1 = 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n'
    instruction_following_2 = 'Remember to put your answer on its own line after "Answer:".'
    # add a row to each data item that represents a unique id
    def make_map_fn(split):

        def process_fn(example, idx):
            question_raw = example.pop('problem')

            question = instruction_following_1 + '\n' + question_raw + '\n' + instruction_following_2

            answer_raw = example.pop('answer')
            data = {
                "data_source": 'math_dapo',
                "prompt": [{
                    "role": "user",
                    "content": question,
                }],
                "ability": "MATH",
                "reward_model": {
                    "style": "rule-lighteval/MATH_v2",
                    "ground_truth": answer_raw
                },
                "extra_info": {
                    'dummy': 'dummy',
                }
            }
            return data

        return process_fn

    dataset = dataset.map(function=make_map_fn('test'), with_indices=True)
    return dataset

math500_dataset = prepare_math500()
math500_dataset.to_parquet('/gpfs/users/zhangyiqi/srl/data/math500_eval.parquet')

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

553774

In [ ]:
def prepare_amc23():
    data_source = '/gpfs/users/zhangyiqi/.cache/huggingface/hub/datasets--math-ai--amc23'

    dataset = datasets.load_dataset(data_source, 'default')

    dataset = dataset['test']

    instruction_following_1 = 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n'
    instruction_following_2 = 'Remember to put your answer on its own line after "Answer:".'
    # add a row to each data item that represents a unique id
    def make_map_fn(split):

        def process_fn(example, idx):
            question_raw = example.pop('question')

            question = instruction_following_1 + '\n' + question_raw + '\n' + instruction_following_2

            answer_raw = int(example.pop('answer'))
            data = {
                "data_source": 'math_dapo',
                "prompt": [{
                    "role": "user",
                    "content": question,
                }],
                "ability": "MATH",
                "reward_model": {
                    "style": "rule-lighteval/MATH_v2",
                    "ground_truth": answer_raw
                },
                "extra_info": {
                    'dummy': 'dummy',
                }
            }
            return data

        return process_fn

    dataset = dataset.map(function=make_map_fn('test'), with_indices=True)
    return dataset

amc23_dataset = prepare_amc23()
amc23_dataset.to_parquet('/gpfs/users/zhangyiqi/srl/data/amc23_eval.parquet')

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

27151

In [22]:
def prepare_minerva():
    data_source = '/gpfs/users/zhangyiqi/.cache/huggingface/hub/datasets--zwhe99--simplerl-minerva-math'

    dataset = datasets.load_dataset(data_source, 'default')

    dataset = dataset['test']

    instruction_following_1 = 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n'
    instruction_following_2 = 'Remember to put your answer on its own line after "Answer:".'
    # add a row to each data item that represents a unique id
    def make_map_fn(split):

        def process_fn(example, idx):
            question_raw = example.pop('problem')

            question = instruction_following_1 + '\n' + question_raw + '\n' + instruction_following_2

            answer_raw = example.pop('answer')
            data = {
                "data_source": 'math_dapo',
                "prompt": [{
                    "role": "user",
                    "content": question,
                }],
                "ability": "MATH",
                "reward_model": {
                    "style": "rule-lighteval/MATH_v2",
                    "ground_truth": answer_raw
                },
                "extra_info": {
                    'dummy': 'dummy',
                }
            }
            return data

        return process_fn

    dataset = dataset.map(function=make_map_fn('test'), with_indices=True)
    return dataset
minerva_dataset = prepare_minerva()
minerva_dataset.to_parquet('/gpfs/users/zhangyiqi/srl/data/minerva_eval.parquet')

Map:   0%|          | 0/272 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

315378

In [ ]:
import os
import torch
import glob
from collections import OrderedDict
import re
import shutil
from pathlib import Path
from accelerate.utils import merge_fsdp_weights

ckpt_dir   = "/mnt/blob/ckpts_bugfix/DAPO-PPO/Qwen2.5-32B/global_step_200/critic"
base_model = "qwen/Qwen2.5-32B"
out_dir    = "/tmp/Qwen-2.5-32B-200-critic"
os.makedirs(out_dir, exist_ok=True)

In [ ]:
import os, re, torch
from collections import defaultdict
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, AutoModelForTokenClassification



# # -------- 1. load every rank checkpoint --------
regex   = re.compile(r"model_world_size_\d+_rank_(\d+)\.pt")
rank_sd = {}                              # rank → state-dict
for f in os.listdir(ckpt_dir):
    m = regex.match(f)
    if m:
        rank = int(m.group(1))
        rank_sd[rank] = torch.load(os.path.join(ckpt_dir, f), map_location="cpu")

world_size = len(rank_sd)
assert world_size > 0, "no rank files found"

# -------- 2. collect per-param slices --------
slices = defaultdict(list)                # param → [local_tensor per rank]
for rank in range(world_size):
    for k, v in rank_sd[rank].items():
        if isinstance(v, torch.distributed._tensor.DTensor):
            # grab the shard stored on this rank
            slices[k].append(v._local_tensor)     # private attr but works fine
        else:                                     # unsharded params / scalars
            slices[k] = [v] * world_size          # replicate so cat() is no-op

# -------- 3. reassemble full tensors --------
full = {}
for k, parts in slices.items():
    # if more than one shard, concatenate along dim 0
    full[k] = torch.cat(parts, dim=0) if len(parts) > 1 else parts[0]

# -------- 4. drop into an HF model and save --------
cfg   = AutoConfig.from_pretrained(base_model)
model = AutoModelForTokenClassification.from_config(cfg)
model.load_state_dict(full, strict=False)
model.save_pretrained(out_dir)
AutoTokenizer.from_pretrained(base_model).save_pretrained(out_dir)

print("✅ merged checkpoint written to", out_dir)


In [9]:
def prepare_olympiad():
    data_source = '/gpfs/users/zhangyiqi/.cache/huggingface/hub/datasets--Hothan--OlympiadBench'

    dataset = datasets.load_dataset(data_source)

    dataset = dataset['train']

    instruction_following_1 = 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n'
    instruction_following_2 = 'Remember to put your answer on its own line after "Answer:".'
    # add a row to each data item that represents a unique id
    def make_map_fn(split):

        def process_fn(example, idx):
            question_raw = example.pop('question')

            question = instruction_following_1 + '\n' + question_raw + '\n' + instruction_following_2

            answer_raw = example.pop('final_answer')[0]
            data = {
                "data_source": 'dapo-math',
                "prompt": [{
                    "role": "user",
                    "content": question,
                }],
                "ability": "MATH",
                "reward_model": {
                    "style": "rule-lighteval/MATH_v2",
                    "ground_truth": answer_raw
                },
                "extra_info": {
                    'dummy': 'dummy',
                }
            }
            return data

        return process_fn

    dataset = dataset.map(function=make_map_fn('test'), with_indices=True)
    dataset = dataset.remove_columns(['solution'])
    # dataset.cast_column("prompt", datasets.Sequence(datasets.Value(dty
    return dataset
olympiad_dataset = prepare_olympiad()
olympiad_dataset.to_parquet('/gpfs/users/zhangyiqi/srl/data/olympiad_eval.parquet')

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

552264

In [12]:
olympiad_dataset.__len__()

674

In [ ]:
# merge all datasets
all_datasets = datasets.concatenate_datasets([gsm8k_dataset, math500_dataset, minerva_dataset, olympiad_dataset])


In [13]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-3.1-8B-Instruct')
processor = AutoProcessor.from_pretrained('meta-llama/Llama-3.1-8B-Instruct')
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/aime-2024.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )

dataset[100]['data_source']

SSLError: (MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /meta-llama/Llama-3.1-8B-Instruct/resolve/main/tokenizer_config.json (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1007)')))"), '(Request ID: 98072fcd-82cd-46cc-9daf-a2f5cc0b9a6e)')

In [20]:
def prepare_amc():
    data_source = '/gpfs/users/zhangyiqi/.cache/huggingface/hub/datasets--math-ai--amc23'

    dataset = datasets.load_dataset(data_source, 'default')

    dataset = dataset['test']
    dataset = dataset.repeat(32)

    instruction_following_1 = 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n'
    instruction_following_2 = 'Remember to put your answer on its own line after "Answer:".'
    # add a row to each data item that represents a unique id
    def make_map_fn(split):

        def process_fn(example, idx):
            question_raw = example.pop('question')

            question = instruction_following_1 + '\n' + question_raw + '\n' + instruction_following_2

            answer_raw = example.pop('answer')
            data = {
                "data_source": 'dapo-math',
                "prompt": [{
                    "role": "user",
                    "content": question,
                }],
                "ability": "MATH",
                "reward_model": {
                    "style": "rule-lighteval/MATH_v2",
                    "ground_truth": answer_raw
                },
                "extra_info": {
                    'dummy': 'dummy',
                }
            }
            return data

        return process_fn

    dataset = dataset.map(function=make_map_fn('test'), with_indices=True)
    return dataset
amc_dataset = prepare_amc()
amc_dataset.to_parquet('/gpfs/users/zhangyiqi/srl/data/amc_eval.parquet')

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

868832

In [21]:
amc_dataset.__len__()

1280